# 📊 **Telco Customer Churn Prediction: End-to-End Machine Learning Pipeline**

## 📖 **1. Problem Statement & Business Context**
Customer churn, also known as customer attrition, occurs when customers stop doing business with a company. For subscription-based service providers (like telecommunications firms), retaining existing customers is significantly less expensive than acquiring new ones. 

By building an predictive model, a telecom operator can identify high-risk accounts and proactively target them with retention incentives (discounts, contract upgrades, custom support), ultimately protecting recurring revenue streams.

## 🎯 **2. Objective**
The main goal of this project is to construct a **production-ready machine learning pipeline** that:
1. **Cleanses and standardizes** the Telco Customer Churn dataset.
2. **Conducts Exploratory Data Analysis (EDA)** to surface customer behaviors linked to churn.
3. **Builds a robust scikit-learn Pipeline** combining feature imputation, scaling, and categorical encoding.
4. **Optimizes model performance** using stratified cross-validated Grid Search (`GridSearchCV`).
5. **Compares and evaluates** Logistic Regression and Random Forest architectures.
6. **Serializes and exports** the end-to-end model pipeline using `joblib` for seamless deployment in production APIs.

---  
## ⚙️ **Step 0: Setup and Environment Configuration**  
First, we load the required packages and configure styling parameters for visualizations.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

import joblib

# Set premium plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 16
})

SEED = 42
np.random.seed(SEED)

---  
## 📥 **Step 1: Data Ingestion with Robust Fallbacks**  
To ensure that this notebook runs out-of-the-box in any execution environment, we implement a robust loading function that searches:
1. **Local paths** (useful when running locally in development workspaces).
2. **Kaggle paths** (under `/kaggle/input` when running on the Kaggle notebook platform).
3. **Kagglehub direct API download** (as an automated, secure fallback to download the Telco dataset directly from Kaggle repository if not present locally).

In [ ]:
def load_telco_data():
    """
    Attempts to load the Telco Customer Churn dataset from multiple sources:
    1. Local workspace path
    2. Kaggle environment path
    3. Automated download using kagglehub
    """
    # 1. Local paths
    local_paths = [
        Path("WA_Fn-UseC_-Telco-Customer-Churn.csv"),
        Path("predicting customer churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"),
        Path("../WA_Fn-UseC_-Telco-Customer-Churn.csv"),
    ]
    for path in local_paths:
        if path.exists():
            print(f"[Local] Loading dataset from path: {path.absolute()}")
            return pd.read_csv(path)
            
    # 2. Kaggle environment path
    kaggle_base = Path("/kaggle/input")
    if kaggle_base.exists():
        print("[Kaggle] Searching input directories for Telco Customer Churn dataset...")
        for root, _, files in os.walk(str(kaggle_base)):
            for f in files:
                if f.lower().endswith(".csv") and "telco" in f.lower():
                    csv_path = Path(root) / f
                    print(f"[Kaggle] Loading dataset from path: {csv_path}")
                    return pd.read_csv(csv_path)
                    
    # 3. Fallback to kagglehub download
    print("[Fallback] Dataset not found locally or in Kaggle inputs. Fetching via kagglehub...")
    try:
        import kagglehub
        download_path = kagglehub.dataset_download("blastchar/telco-customer-churn")
        print(f"[Fallback] Downloaded dataset files to: {download_path}")
        for file in Path(download_path).glob("*.csv"):
            print(f"[Fallback] Loading downloaded CSV: {file}")
            return pd.read_csv(file)
    except Exception as e:
        print(f"[Fallback] Kagglehub download failed: {e}")
        
    raise FileNotFoundError("Unable to locate or download the Telco Customer Churn dataset.")

# Load dataset
df = load_telco_data()
print("Loaded Dataset Shape:", df.shape)
display(df.head())

---  
## 🧹 **Step 2: Data Cleaning & Type Formatting**  
A clean dataset is critical for successful ML modeling. We address several data quality issues:
1. **Column Whitespace**: Remove any trailing spaces from the column labels.
2. **TotalCharges Formatting**: The column `TotalCharges` is parsed as an `object` type because it contains blank string spaces (`" "`). These represent new accounts with `tenure = 0` who have not completed their first billing cycle. Thus, their total charge should logically be `0.0`. We coerce these spaces to `NaN` and impute them with `0.0`.
3. **Target Variable Mapping**: Map the target column `Churn` (`Yes` / `No`) to numeric binary values (`1` / `0`).
4. **Removing Extraneous Attributes**: Drop `customerID` since unique identifiers provide no predictive utility and lead to overfitting.

In [ ]:
# Clean column names
df.columns = [c.strip() for c in df.columns]

# Display dataset summary info
print("--- Column Types and Raw Summaries ---")
df.info()

# Address TotalCharges spaces
spaces = (df["TotalCharges"] == " ").sum()
print(f"\nTotalCharges rows containing blank spaces: {spaces}")

if spaces > 0:
    # Inspect rows with blank charges to verify tenure relationship
    print("Preview of rows with blank TotalCharges (confirming tenure=0):")
    display(df[df["TotalCharges"] == " "][["tenure", "MonthlyCharges", "TotalCharges"]].head())

# Convert to numeric, forcing spaces to NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Impute missing TotalCharges with 0.0
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)
print("TotalCharges successfully converted and cleaned.")

# Encode Churn target
if "Churn" in df.columns:
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
    print("Target 'Churn' mapped to: Yes -> 1, No -> 0")

# Drop identifier columns
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])
    print("Dropped 'customerID' identifier.")

print(f"\nFinal Shape after cleaning: {df.shape}")
print("Null counts after cleaning:")
print(df.isnull().sum())

---  
## 📊 **Step 3: Exploratory Data Analysis (EDA)**  
We construct high-quality plots to uncover factors driving churn:
1. **Churn Distribution**: Quantify class imbalance.
2. **Numerical Distributions (Tenure & Monthly Charges)**: Contrast metrics for active vs. churned customers.
3. **Contract Types**: Highlight the correlation between Month-to-Month contracts and churn.
4. **Payment Methods**: Explore churn rates based on payment choice.
5. **Multicollinearity Heatmap**: Visualizing correlation between numerical features.

In [ ]:
# Set premium aesthetic settings
colors = ["#1f77b4", "#ff7f0e"]

# 1. Class Distribution
plt.figure(figsize=(6, 5))
ax = sns.countplot(x="Churn", data=df, palette=colors)
plt.title("Class Distribution (Churn Imbalance Check)", pad=15)
plt.xlabel("Churn Status (0 = Retained, 1 = Churned)")
plt.ylabel("Count")

total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height()/total:.1f}%'
    x_pos = p.get_x() + p.get_width() / 2
    y_pos = p.get_height() + 50
    ax.annotate(percentage, (x_pos, y_pos), ha='center', va='bottom', fontweight='bold', size=11)

plt.tight_layout()
plt.show()

# 2. Tenure & Monthly Charges Stratified by Target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(data=df, x="tenure", hue="Churn", fill=True, common_norm=False, alpha=0.4, palette=colors, ax=axes[0])
axes[0].set_title("Customer Tenure Distribution by Churn Status")
axes[0].set_xlabel("Tenure (Months)")
axes[0].set_ylabel("Density")

sns.kdeplot(data=df, x="MonthlyCharges", hue="Churn", fill=True, common_norm=False, alpha=0.4, palette=colors, ax=axes[1])
axes[1].set_title("Monthly Charges Distribution by Churn Status")
axes[1].set_xlabel("Monthly Charges ($)")
axes[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

# 3. Contract Type Churn Rates
plt.figure(figsize=(8, 5))
ax_contract = sns.barplot(x="Contract", y="Churn", data=df, errorbar=None, order=["Month-to-month", "One year", "Two year"], palette="Blues_r")
plt.title("Average Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (Percentage)")
for p in ax_contract.patches:
    rate = f'{100 * p.get_height():.1f}%'
    ax_contract.annotate(rate, (p.get_x() + p.get_width()/2, p.get_height() + 0.01), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

# 4. Payment Method Churn Rates
plt.figure(figsize=(10, 5))
ax_pay = sns.barplot(x="Churn", y="PaymentMethod", data=df, errorbar=None, palette="Oranges_r")
plt.title("Churn Distribution by Payment Method")
plt.xlabel("Average Churn Rate")
plt.ylabel("Payment Method")
plt.tight_layout()
plt.show()

# 5. Correlation Heatmap of Numerical Features
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()

---  
## ⚙️ **Step 4: Train-Test Split & Leakage Prevention**  
To validate model generalization correctly and prevent **data leakage**, we split the dataset into training (80%) and testing (20%) subsets *prior* to scaling or encoding. 

We use `stratify=y` to ensure both subsets maintain the same ratio of churned vs. retained customers, which is critical due to the target class imbalance.

In [ ]:
# Separate features and target
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f"Training instances: {X_train.shape[0]} samples")
print(f"Testing instances: {X_test.shape[0]} samples")
print(f"Train churn proportion: {y_train.mean():.2%}")
print(f"Test churn proportion: {y_test.mean():.2%}")

---  
## 🛠️ **Step 5: Preprocessing Pipeline Construction**  
Instead of transforming the entire dataframe manually, we structure an automated scikit-learn preprocessing pipeline:
- **Numerical Sub-Pipeline**: Median Imputation + Standardization (`StandardScaler`).
- **Categorical Sub-Pipeline**: Mode Imputation + One-Hot Encoding (`OneHotEncoder`). We use `drop='first'` to prevent multicollinearity (which can degrade linear model performance like Logistic Regression).

In [ ]:
# Identify numerical and categorical columns automatically
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numerical columns to process:", numeric_features)
print("Categorical columns to process:", categorical_features)

# Safe instantiation of OneHotEncoder supporting sparse vs. sparse_output flags
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")
except TypeError:
    # Compatibility fallback for older scikit-learn versions
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False, drop="first")

# Numeric preprocessing steps
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing steps
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print("\nColumnTransformer preprocessing pipeline ready.")

---  
## 🚀 **Step 6: Hyperparameter Tuning via GridSearchCV**  
We train and tune two distinct algorithms to find the best classifier:
1. **Logistic Regression** (baseline linear model, highly interpretable).
2. **Random Forest** (ensemble tree-based model, handles complex features interactions).

We use **5-fold Stratified Cross-Validation** and optimize for the **F1-Score** metric to evaluate each parameter combination, balancing Precision and Recall correctly.

In [ ]:
# Stratified K-Fold setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print("=== Tuning Logistic Regression ===")
log_reg_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=SEED))
])

log_reg_param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__solver": ["liblinear", "lbfgs"]
}

log_reg_grid = GridSearchCV(
    log_reg_pipe,
    param_grid=log_reg_param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

log_reg_grid.fit(X_train, y_train)
print(f"Best Logistic Regression parameters: {log_reg_grid.best_params_}")
print(f"Best Cross-Validation F1-score: {log_reg_grid.best_score_:.4f}")

In [ ]:
print("=== Tuning Random Forest ===")
rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=SEED))
])

rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10, 15, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    rf_pipe,
    param_grid=rf_param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)
print(f"Best Random Forest parameters: {rf_grid.best_params_}")
print(f"Best Cross-Validation F1-score: {rf_grid.best_score_:.4f}")

---  
## 📊 **Step 7: Evaluation & Model Comparison**  
We validate the tuned estimators on the unseen testing dataset using five metrics: Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

In [ ]:
best_models = {
    "Logistic Regression": log_reg_grid.best_estimator_,
    "Random Forest": rf_grid.best_estimator_
}

results = []

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False)
print("\n--- Model Performance Comparison (Test Dataset) ---")
display(results_df.round(4))

---  
## 📈 **Step 8: Detailed Best Model Assessment**  
We display the detailed classification report, confusion matrix, and ROC-AUC curve of our best-performing pipeline.

In [ ]:
# Select model with highest F1 score
best_name = results_df.iloc[0]["Model"]
best_model = best_models[best_name]

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(f"Selected Best Model: {best_name}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Plot Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0], annot_kws={"size": 12, "weight": "bold"})
axes[0].set_title(f"Confusion Matrix - {best_name}", pad=10)
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")
axes[0].set_xticklabels(["Retained", "Churned"])
axes[0].set_yticklabels(["Retained", "Churned"])

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_val = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color="darkorange", lw=2.5, label=f"ROC (AUC = {auc_val:.4f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title(f"ROC Curve - {best_name}", pad=10)
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

---  
## 🔍 **Step 9: Feature Importance & Interpretability**  
Understanding *why* a model predicts churn is crucial for business action. Here we extract feature importances (for Random Forest) or coefficients (for Logistic Regression) from the pipeline.

In [ ]:
# Extract features
preprocessor_step = best_model.named_steps["preprocessor"]
model_step = best_model.named_steps["model"]

# Get one-hot feature names
cat_encoder = preprocessor_step.named_transformers_["cat"].named_steps["onehot"]
encoded_cat_features = cat_encoder.get_feature_names_out(categorical_features).tolist()
all_features = numeric_features + encoded_cat_features

if best_name == "Random Forest":
    # Sort feature importances
    importances = model_step.feature_importances_
    feat_imp = pd.DataFrame({
        "Feature": all_features,
        "Importance": importances
    }).sort_values("Importance", ascending=False)
    
    print("Top 15 Most Important Features:")
    display(feat_imp.head(15))
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feat_imp.head(15), x="Importance", y="Feature", palette="viridis")
    plt.title("Top 15 Feature Importances (Random Forest Model)")
    plt.xlabel("Mean Decrease in Impurity (MDI)")
    plt.ylabel("Feature Name")
    plt.tight_layout()
    plt.show()

elif best_name == "Logistic Regression":
    # Sort coefficients
    coefficients = model_step.coef_[0]
    feat_coef = pd.DataFrame({
        "Feature": all_features,
        "Coefficient": coefficients,
        "Absolute Coefficient": np.abs(coefficients)
    }).sort_values("Absolute Coefficient", ascending=False)
    
    print("Top 15 Most Influential Features:")
    display(feat_coef.head(15))
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feat_coef.head(15), x="Coefficient", y="Feature", palette="coolwarm")
    plt.axvline(0, color="black", linestyle="--", linewidth=1)
    plt.title("Top 15 Feature Coefficients (Logistic Regression Model)")
    plt.xlabel("Coefficient Value (Positive indicates higher risk of churn)")
    plt.ylabel("Feature Name")
    plt.tight_layout()
    plt.show()

---  
## 💾 **Step 10: Exporting the Production Pipeline**  
To make our model production-ready, we serialize the **entire pipeline object** (preprocessor + model step) into a single `.joblib` file. This guarantees that new, raw records can be predicted on directly without manual scaling or hot-encoding steps.

In [ ]:
# Define output directory paths with Kaggle/local fallbacks
output_dir = Path("/kaggle/working/telco_churn_pipeline")
if not Path("/kaggle/working").exists():
    output_dir = Path("./telco_churn_pipeline")

output_dir.mkdir(parents=True, exist_ok=True)
pipeline_file = output_dir / "telco_churn_pipeline.joblib"

# Export pipeline
joblib.dump(best_model, pipeline_file)
print(f"Production-ready pipeline successfully saved to: {pipeline_file.absolute()}")

---  
## 💻 **Step 11: Production Inference Simulation**  
Here we simulate a live web API endpoint. We load our exported pipeline, ingest a completely raw client data payload (with missing fields or original formats), and perform a prediction.

In [ ]:
# 1. Load exported pipeline
production_pipeline = joblib.load(pipeline_file)
print("Loaded production pipeline successfully.")

# 2. Mock client data (completely raw dictionary representing a single customer)
mock_customer_raw = pd.DataFrame([{
    "gender": "Female",
    "SeniorCitizen": 1,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 2,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "Yes",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 70.05,
    "TotalCharges": 140.10
}])

print("\n--- Incoming Raw Customer API Payload ---")
display(mock_customer_raw)

# 3. Run prediction directly
pred_class = production_pipeline.predict(mock_customer_raw)[0]
pred_prob = production_pipeline.predict_proba(mock_customer_raw)[0, 1]

print("\n--- Live Inference Engine Output ---")
print(f"Predicted Class: {pred_class} ({'Churn Risk' if pred_class == 1 else 'Loyal'})")
print(f"Probability of Churn: {pred_prob:.2%}")

---  
## 🏁 **Step 12: Explanation of Results & Final Insights**  

### 📈 **Core Findings & Business Implications**
1. **Contract Structure**: Customers on **Month-to-month** contracts show a drastically higher churn rate compared to those on one-year or two-year contracts. Proactive marketing campaigns targeting these users with long-term subscription discounts would mitigate a major churn vector.
2. **Payment Type**: Electronic check payers demonstrate an elevated churn propensity. Setting up promotional credits for transitioning to Credit Card or Bank Transfer autopay could decrease churn.
3. **Tenure Relationship**: Churn is concentrated in the early months of subscription (0 to 12 months). A robust onboarding phase is critical to retaining accounts past their first year.

### ⚙️ **Engineering Achievements**
- **Feature Drift Mitigation**: By encapsulating all scaling, mode/median imputations, and dummy encodings in a single `ColumnTransformer` + `Pipeline`, we guarantee that new input features are handled exactly the same way during offline validation and online web inference.
- **Robust Evaluation**: We tuned our models for F1-score rather than raw accuracy. Under target imbalance (26.5% churn), optimizing for accuracy yields models that predict "no churn" everywhere. F1-score optimization ensures we identify high-risk churners effectively while managing retention spend budget.